# Assignment PAIRS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm


pd.set_option('display.max_columns', None) # Show all columns when printing a dataframe
pd.set_option('display.max_rows', None)

## Load the workbook

In [ ]:
df = pd.read_excel('/Users/jlaw/projects/stern/systematic-investing/data/Assignment_PAIRS_data.xlsx', sheet_name='data', header=6)
df[['Date', 'ticker1', 'closeWHITE', 'ticker2', 'closeBLACK']].head()

## Dickey Fuller test

In [ ]:
df['spread_price'] = df['closeWHITE'] - df['closeBLACK']
df['delta_spread'] = df['spread_price'].diff()

df4years = df.iloc[:1000].copy() # don't mess with the original dataset, take a copy
df_test = df4years[['Date', 'spread_price', 'delta_spread']].copy()
df_test['lag_spread_price'] = df_test['spread_price'].shift(1)
df_test = df_test.dropna().reset_index(drop=True)
df_test.head()

In [ ]:
x = df_test['lag_spread_price']
y = df_test['delta_spread']

X = sm.add_constant(x)
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
critical_values = {
    '2%': -3.43,
    '5%': -3.12,
    '10%': -2.57,
}

for level, critical_value in critical_values.items():
    if model.tvalues['lag_spread_price'] < critical_value:
        print(f"Reject null hypothesis at the {level} level.")
    else:
        print(f"Fail to reject null hypothesis at the {level} level.")

## Dickey Fuller interpretation:
### There is weak evidence to showing stationarity of the spread
### This is a marginal suggestion that the spread is mean reverting
#### (verus remaining permanently affected)

## Build the Pairs strategy signals

In [ ]:

df['ret1WHITE'] = df['closeWHITE'].pct_change()
df['ret1BLACK'] = df['closeBLACK'].pct_change()

def zscore(df, ticker):
    for horizon in [5, 10, 20]:
        df[f'{ticker}_ret{horizon}'] = df[f'close{ticker}'].pct_change(horizon)
        df[f'z{ticker}{horizon}'] = (
            (df[f'{ticker}_ret{horizon}'] - df[f'{ticker}_ret{horizon}'].shift(1).rolling(60).mean())
            / df[f'{ticker}_ret{horizon}'].shift(1).rolling(60).std()
        )

zscore(df, 'WHITE')
zscore(df, 'BLACK')

for horizon in [5, 10, 20]:
    df[f'zdiff{horizon}'] = df[f'zWHITE{horizon}'] - df[f'zBLACK{horizon}']

df['vol20WHITE'] = df['ret1WHITE'].rolling(20).std()
df['vol20BLACK'] = df['ret1BLACK'].rolling(20).std()
df['WtWHITE'] = 1 / df['vol20WHITE']
df['wtBLACK'] = 1 / df['vol20BLACK']
df['sumwts'] = df['WtWHITE'] + df['wtBLACK']
df['wWHITE'] = df['WtWHITE'] / df['sumwts']
df['wBLACK'] = df['wtBLACK'] / df['sumwts']
df['FRet1'] = df['wWHITE'] * df['ret1WHITE'].shift(-1) - df['wBLACK'] * df['ret1BLACK'].shift(-1)

# df.head(100)

## Run the strategy

In [ ]:
def run_strategy(spreadCol, fret1, long_entry=-1, short_entry=1, long_cap=1, short_cap=-1, max_holding=5):
    age = 0
    ages = []
    position = []
    current_position = 0

    for i, spread in enumerate(spreadCol):
        if current_position == 1 and (spread >= long_cap or age >= max_holding):
            current_position = 0 # exit long
            age = 0
        elif current_position == -1 and (spread <= short_cap or age >= max_holding):
            current_position = 0 # exit short
            age = 0

        if current_position == 0:
            if spread <= long_entry:
                current_position = 1 # go long
                age = 1
            elif spread >= short_entry:
                current_position = -1 # go short
                age = 1
        else:
            age += 1
        
        ages.append(age)
        position.append(current_position)
    
    results = pd.DataFrame({
        'position': position,
        'age': ages
    })
    results['strategy_return'] = results['position'] * fret1
    results['strategy_cumulative_return'] = (1 + results['strategy_return'].fillna(0)).cumprod()
    return results

result5 = run_strategy(df['zdiff5'], df['FRet1'])
result10 = run_strategy(df['zdiff10'], df['FRet1'])
result20 = run_strategy(df['zdiff20'], df['FRet1'])

## Evaluate strategy

In [ ]:
print('5 Day Returns Spread strategy Sharpe Ratio:', result5['strategy_return'].mean() / result5['strategy_return'].std() * np.sqrt(260))
print('10 Day Returns Spread strategy Sharpe Ratio:', result10['strategy_return'].mean() / result10['strategy_return'].std() * np.sqrt(260))
print('20 Day Returns Spread strategy Sharpe Ratio:', result20['strategy_return'].mean() / result20['strategy_return'].std() * np.sqrt(260))

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], result5['strategy_cumulative_return'], label='5 Day Spread Strategy')
plt.plot(df['Date'], result10['strategy_cumulative_return'], label='10 Day Spread Strategy')
plt.plot(df['Date'], result20['strategy_cumulative_return'], label='20 Day Spread Strategy')
plt.title('Cumulative Returns of Spread Strategies')
plt.xlabel('Time')
plt.ylabel('Cumulative Return')
plt.legend()
plt.show()

# Analysis

### The 5-Day and 20-Day strategies have similar Sharpe and cumulative returns.
### Despite this, the Sharpe and performance of each is unconvincing, and I would probably not run the strategy with my own money.
### I would certainly not trade the 10-Day strategy becuase it is inferior to the others.